# Binance Spot BTC quote-pair kline coverage

This notebook reproduces the exploratory workbook that lists every Binance Spot symbol whose quote asset is BTC and records the earliest kline returned for six intervals.

Methodological choices:

- retain both `TRADING` and `BREAK` symbols returned by `exchangeInfo` to preserve historical coverage;
- use `1d`, `1s`, `1m`, `1h`, `1w`, and `1M`;
- request one record from `startTime=0`;
- use UTC as the canonical timezone;
- write the workbook to `data/exploration/`.

Required packages: `pandas` and `openpyxl`. The Binance public market-data endpoints do not require an API key.

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

BASE_URL = "https://api.binance.com"
EXCHANGE_INFO_PATH = "/api/v3/exchangeInfo"
KLINES_PATH = "/api/v3/klines"
QUOTE_ASSET = "BTC"
INTERVALS = ("1d", "1s", "1m", "1h", "1w", "1M")
REQUEST_TIMEOUT_SECONDS = 30
MAX_RETRIES = 6
MAX_WORKERS = 20

COLUMN_BY_INTERVAL = {
    "1d": "First Daily Kline (1d, UTC)",
    "1s": "First Second Kline (1s, UTC)",
    "1m": "First Minute Kline (1m, UTC)",
    "1h": "First Hour Kline (1h, UTC)",
    "1w": "First Week Kline (1w, UTC)",
    "1M": "First Month Kline (1M, UTC)",
}


def find_repository_root(start: Path | None = None) -> Path:
    """Find the repository root by looking for AGENTS.md."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root()
OUTPUT_PATH = (
    REPOSITORY_ROOT
    / "data"
    / "exploration"
    / "binance_spot_btc_quote_pair_kline_coverage.xlsx"
)
OUTPUT_PATH

In [ ]:
def request_json(
    path: str,
    parameters: dict[str, object] | None = None,
) -> object:
    """Request JSON from Binance with bounded retries for transient errors."""
    query_string = urlencode(parameters or {})
    url = f"{BASE_URL}{path}"
    if query_string:
        url = f"{url}?{query_string}"

    for attempt in range(MAX_RETRIES + 1):
        request = Request(
            url,
            headers={"User-Agent": "tcc-crypto-research/0.1"},
        )
        try:
            with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
                return json.load(response)
        except HTTPError as error:
            is_transient = error.code == 429 or 500 <= error.code < 600
            if is_transient and attempt < MAX_RETRIES:
                retry_after = error.headers.get("Retry-After")
                delay_seconds = (
                    float(retry_after)
                    if retry_after is not None
                    else min(2**attempt, 30)
                )
                time.sleep(delay_seconds)
                continue
            response_body = error.read().decode("utf-8", errors="replace")
            raise RuntimeError(
                f"Binance returned HTTP {error.code} for {url}: {response_body}"
            ) from error
        except URLError as error:
            if attempt < MAX_RETRIES:
                time.sleep(min(2**attempt, 30))
                continue
            raise RuntimeError(f"Could not reach Binance at {url}: {error.reason}") from error

    raise RuntimeError(f"Request retry loop ended unexpectedly for {url}.")


def get_btc_quote_pairs() -> pd.DataFrame:
    """Return Spot-enabled symbols that use BTC as the quote asset."""
    payload = request_json(EXCHANGE_INFO_PATH)
    if not isinstance(payload, dict) or not isinstance(payload.get("symbols"), list):
        raise ValueError("Unexpected Binance exchangeInfo response.")

    pairs = [
        {
            "Trading Pair": item["symbol"],
            "Base Asset": item["baseAsset"],
            "Quote Asset": item["quoteAsset"],
            "Status": item["status"],
        }
        for item in payload["symbols"]
        if item.get("quoteAsset") == QUOTE_ASSET
        and item.get("isSpotTradingAllowed", False)
    ]

    if not pairs:
        raise ValueError(f"No Spot symbols quoted in {QUOTE_ASSET} were returned.")

    return pd.DataFrame(pairs)


def get_earliest_kline_open_time(symbol: str, interval: str) -> pd.Timestamp:
    """Return the opening timestamp of the earliest kline from the API."""
    payload = request_json(
        KLINES_PATH,
        {
            "symbol": symbol,
            "interval": interval,
            "startTime": 0,
            "limit": 1,
            "timeZone": 0,
        },
    )
    if not isinstance(payload, list) or not payload:
        raise ValueError(f"No {interval} kline was returned for {symbol}.")
    if not isinstance(payload[0], list) or not payload[0]:
        raise ValueError(f"Unexpected {interval} kline response for {symbol}.")
    return pd.to_datetime(payload[0][0], unit="ms", utc=True)

In [ ]:
pairs = get_btc_quote_pairs()
print(f"Pairs returned: {len(pairs):,}")
pairs["Status"].value_counts()

In [ ]:
jobs = [
    (symbol, interval)
    for symbol in pairs["Trading Pair"]
    for interval in INTERVALS
]
earliest_by_symbol: dict[str, dict[str, pd.Timestamp]] = {
    symbol: {} for symbol in pairs["Trading Pair"]
}
errors: list[dict[str, str]] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(get_earliest_kline_open_time, symbol, interval): (
            symbol,
            interval,
        )
        for symbol, interval in jobs
    }

    for completed, future in enumerate(as_completed(futures), start=1):
        symbol, interval = futures[future]
        try:
            earliest_by_symbol[symbol][interval] = future.result()
        except Exception as error:
            errors.append(
                {
                    "Trading Pair": symbol,
                    "Interval": interval,
                    "Error": str(error),
                }
            )

        if completed % 250 == 0 or completed == len(jobs):
            print(f"Completed {completed:,}/{len(jobs):,} requests")

if errors:
    error_frame = pd.DataFrame(errors)
    display(error_frame)
    raise RuntimeError(
        f"Collection stopped because {len(errors)} request(s) failed. "
        "Review the error table and rerun the cell."
    )

In [ ]:
coverage = pairs.copy()
for interval, column_name in COLUMN_BY_INTERVAL.items():
    coverage[column_name] = coverage["Trading Pair"].map(
        lambda symbol: earliest_by_symbol[symbol].get(interval)
    )

date_columns = list(COLUMN_BY_INTERVAL.values())
coverage["Missing Intervals"] = coverage[date_columns].isna().sum(axis=1)
coverage["Source URL"] = f"{BASE_URL}{KLINES_PATH}"
coverage = coverage.sort_values(
    by=["First Daily Kline (1d, UTC)", "Trading Pair"],
    na_position="last",
).reset_index(drop=True)

coverage.head()

In [ ]:
def apply_clean_workbook_style(writer: pd.ExcelWriter, row_count: int) -> None:
    """Apply simple formatting without changing the underlying data."""
    header_fill = PatternFill("solid", fgColor="17365D")
    header_font = Font(color="FFFFFF", bold=True)
    label_fill = PatternFill("solid", fgColor="F3F6F9")
    section_fill = PatternFill("solid", fgColor="D9EAF7")
    thin_gray = Side(style="thin", color="E5E7EB")

    coverage_sheet = writer.sheets["Kline Coverage"]
    coverage_sheet.sheet_view.showGridLines = False
    coverage_sheet.freeze_panes = "B2"
    coverage_sheet.auto_filter.ref = coverage_sheet.dimensions
    coverage_sheet.row_dimensions[1].height = 36

    for cell in coverage_sheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True
        )

    column_widths = {
        "A": 16, "B": 12, "C": 12, "D": 11,
        "E": 24, "F": 24, "G": 24, "H": 24,
        "I": 24, "J": 24, "K": 16, "L": 48,
    }
    for column_letter, width in column_widths.items():
        coverage_sheet.column_dimensions[column_letter].width = width

    for row in coverage_sheet.iter_rows(min_row=2, max_row=row_count + 1):
        for cell in row:
            cell.border = Border(bottom=thin_gray)
            cell.alignment = Alignment(vertical="center")
        for cell in row[4:10]:
            cell.number_format = "yyyy-mm-dd hh:mm:ss"
            cell.alignment = Alignment(horizontal="center")
        row[10].alignment = Alignment(horizontal="center")
        row[11].hyperlink = row[11].value
        row[11].style = "Hyperlink"

    methodology_sheet = writer.sheets["Methodology"]
    methodology_sheet.sheet_view.showGridLines = False
    methodology_sheet.freeze_panes = "A4"
    methodology_sheet.merge_cells("A1:B1")
    methodology_sheet["A1"] = "Binance Spot BTC Quote-Pair Kline Coverage"
    methodology_sheet["A1"].fill = header_fill
    methodology_sheet["A1"].font = Font(color="FFFFFF", bold=True, size=15)
    methodology_sheet["A1"].alignment = Alignment(vertical="center")
    methodology_sheet.row_dimensions[1].height = 30
    methodology_sheet.column_dimensions["A"].width = 31
    methodology_sheet.column_dimensions["B"].width = 100

    for cell in methodology_sheet[3]:
        cell.fill = section_fill
        cell.font = Font(color="17365D", bold=True)
    for row in methodology_sheet.iter_rows(min_row=4):
        row[0].fill = label_fill
        row[0].font = Font(color="334155", bold=True)
        for cell in row:
            cell.border = Border(bottom=thin_gray)
            cell.alignment = Alignment(vertical="top", wrap_text=True)


def export_coverage_workbook(data: pd.DataFrame, output_path: Path) -> Path:
    """Export coverage data and reproducibility notes to an Excel workbook."""
    collected_at_utc = datetime.now(timezone.utc)
    export_data = data.copy()

    # Excel does not support timezone-aware datetimes. Column labels retain UTC.
    for column_name in COLUMN_BY_INTERVAL.values():
        export_data[column_name] = export_data[column_name].dt.tz_localize(None)

    methodology = pd.DataFrame(
        [
            ("Collection timestamp (UTC)", collected_at_utc.strftime("%Y-%m-%d %H:%M:%S")),
            ("Universe definition", "Binance Spot symbols with BTC as quoteAsset and Spot trading allowed by exchangeInfo."),
            ("Pairs returned", len(export_data)),
            ("TRADING status", int((export_data["Status"] == "TRADING").sum())),
            ("BREAK status", int((export_data["Status"] == "BREAK").sum())),
            ("Intervals queried", ", ".join(INTERVALS)),
            ("Timezone", "UTC (timeZone=0)"),
            ("Request parameters", "startTime=0; limit=1 for every symbol and interval"),
            ("Date meaning", "Opening timestamp of the earliest kline returned for that symbol and interval; it is not necessarily the exact listing or first-trade timestamp."),
            ("Status treatment", "TRADING and BREAK symbols are retained to preserve historical coverage and reduce survivorship bias."),
            ("Exchange information endpoint", f"{BASE_URL}{EXCHANGE_INFO_PATH}"),
            ("Kline endpoint", f"{BASE_URL}{KLINES_PATH}"),
            ("Official documentation", "https://developers.binance.com/en/docs/binance-spot-api-docs/rest-api/market-data-endpoints#klinecandlestick-data"),
        ],
        columns=["Field", "Value"],
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
        datetime_format="yyyy-mm-dd hh:mm:ss",
    ) as writer:
        export_data.to_excel(writer, sheet_name="Kline Coverage", index=False)
        methodology.to_excel(
            writer, sheet_name="Methodology", index=False, startrow=2
        )
        apply_clean_workbook_style(writer, row_count=len(export_data))

    return output_path

In [ ]:
generated_path = export_coverage_workbook(coverage, OUTPUT_PATH)
print(f"Workbook created: {generated_path}")

## Interpretation notes

The timestamps are kline opening boundaries, not exact listing timestamps. Weekly and monthly opening times can therefore precede the first recorded trade within those periods.

The symbol universe is a snapshot of `exchangeInfo` at execution time. Retaining `BREAK` symbols helps preserve historical coverage, but it does not by itself reconstruct which symbols were investable at every historical timestamp. That requires a separate point-in-time eligibility rule.